# Exploratory Data Analysis & Statistics

Run this cell to calculate the Pearson and Spearman correlations.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

BASE_DIR = r"f:\CodeBlocks20.03-AnxietyMonitor"
DATA_FILE = os.path.join(BASE_DIR, "data", "processed", "master_dataset.csv")
RESULTS_DIR = os.path.join(BASE_DIR, "results", "visualizations")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Loading data...")
df = pd.read_csv(DATA_FILE)

# Data Cleaning for analysis
df['survey_reported_stress'] = pd.to_numeric(df['survey_reported_stress'], errors='coerce')
df['anxiety_score'] = pd.to_numeric(df['anxiety_score'], errors='coerce')
df = df.dropna(subset=['survey_reported_stress', 'anxiety_score'])

print(f"Data loaded. Valid rows for analysis: {len(df)}")

# 1. Statistical Correlation
# Aggregate by student to compare their average system score vs their single survey score
student_agg = df.groupby('student_id').agg({
    'anxiety_score': 'mean',
    'survey_reported_stress': 'first',
    'typing_speed_wpm': 'mean',
    'focus_switches': 'max'
}).reset_index()

pearson_corr, p_value_p = pearsonr(student_agg['anxiety_score'], student_agg['survey_reported_stress'])
spearman_corr, p_value_s = spearmanr(student_agg['anxiety_score'], student_agg['survey_reported_stress'])

print("\n--- Statistical Validation ---")
print(f"Pearson Correlation: {pearson_corr:.3f} (p-value: {p_value_p:.4f})")
print(f"Spearman Correlation: {spearman_corr:.3f} (p-value: {p_value_s:.4f})")

with open(os.path.join(BASE_DIR, "results", "statistical_summary.txt"), "w") as f:
    f.write("--- Statistical Validation ---\n")
    f.write(f"Pearson Correlation (Average System Anxiety vs Self-Reported): {pearson_corr:.3f} (p-value: {p_value_p:.4f})\n")
    f.write(f"Spearman Correlation (Average System Anxiety vs Self-Reported): {spearman_corr:.3f} (p-value: {p_value_s:.4f})\n")

# 2. Visualizations
print("\nGenerating Visualizations...")
sns.set_theme(style="whitegrid")

# Plot 1: Scatter plot
plt.figure(figsize=(8, 6))
sns.regplot(x='anxiety_score', y='survey_reported_stress', data=student_agg)
plt.title("System Anxiety Score vs Self-Reported Stress")
plt.xlabel("Average Computed Anxiety Score (0-100)")
plt.ylabel("Self-Reported Stress (1-5)")
plt.savefig(os.path.join(RESULTS_DIR, "anxiety_vs_stress_scatter.png"))
plt.close()

# Plot 2: Correlation Heatmap
numeric_cols = ['typing_speed_wpm', 'latency_variance_ms', 'pause_ratio', 
                'backspace_rate', 'idle_ratio', 'focus_switches', 
                'compile_success_rate', 'session_fragmentation', 
                'anxiety_score', 'survey_reported_stress']

# Convert columns to numeric if needed
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Behavioral Metrics Correlation Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "correlation_heatmap.png"))
plt.close()

print(f"Visualizations saved to {RESULTS_DIR}")


# 10x10 Repeated Cross-Validation Model Training

Run this cell to train the models 100 times each and view the stability.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r"f:\CodeBlocks20.03-AnxietyMonitor"
DATA_FILE = os.path.join(BASE_DIR, "data", "processed", "master_dataset.csv")

print("Loading data for 10x10 Repeated Cross-Validation...")
df = pd.read_csv(DATA_FILE)

features = ['typing_speed_wpm', 'latency_variance_ms', 'pause_ratio',
            'backspace_rate', 'idle_ratio', 'focus_switches',
            'compile_success_rate', 'session_fragmentation', 'anxiety_score']

# Convert columns
for col in features + ['survey_reported_stress']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=features + ['survey_reported_stress'])

# Aggregate per student
student_agg = df.groupby(['student_id', 'session_name']).agg({
    'typing_speed_wpm': 'mean', 'latency_variance_ms': 'mean',
    'pause_ratio': 'mean', 'backspace_rate': 'mean', 'idle_ratio': 'mean',
    'focus_switches': 'max', 'compile_success_rate': 'mean',
    'session_fragmentation': 'mean', 'survey_reported_stress': 'first'
}).reset_index()

# Define features and target (High Stress >= 4)
feat_cols = ['typing_speed_wpm', 'latency_variance_ms', 'pause_ratio',
             'backspace_rate', 'idle_ratio', 'focus_switches',
             'compile_success_rate', 'session_fragmentation']
X = student_agg[feat_cols]
y = (student_agg['survey_reported_stress'] >= 4).astype(int)

print(f"Dataset ready. Total Students: {len(X)}")

# Define models
models = {
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=3),
    'SVM (Linear)': SVC(kernel='linear', random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

# 10 splits, repeated 10 times = 100 training runs per model
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=42)
smote = SMOTE(k_neighbors=3, random_state=42)

print("\nExecuting 10x10 Repeated Cross Validation with SMOTE (100 runs per model)...")
print("-" * 65)
print(f"{'Model':<25} | {'Mean Accuracy':<15} | {'Std Dev (Stability)':<20}")
print("-" * 65)

results = []
for name, model in models.items():
    # SMOTE is inside the pipeline to prevent data leakage during folds
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', smote),
        ('clf', model)
    ])
    
    cv_scores = cross_validate(pipe, X, y, cv=rskf, scoring='accuracy', n_jobs=-1)
    acc_scores = cv_scores['test_score']
    
    mean_acc = np.mean(acc_scores)
    std_acc = np.std(acc_scores)
    
    results.append({
        'Model': name,
        'Mean Accuracy': mean_acc,
        'Std Dev': std_acc
    })
    
    print(f"{name:<25} | {mean_acc*100:>6.2f}%         | ±{std_acc*100:>5.2f}%")

print("-" * 65)

# Save to file
results_df = pd.DataFrame(results).sort_values('Mean Accuracy', ascending=False)
out_path = os.path.join(BASE_DIR, "results", "repeated_10x_cv_results.txt")
with open(out_path, 'w') as f:
    f.write("--- 10x10 Repeated Stratified Cross-Validation (WITH SMOTE) ---\n")
    f.write("Evaluation: 10 Folds, Repeated 10 Times (100 training runs per model)\n")
    f.write("This proves the stability and robustness of the models.\n\n")
    f.write(f"{'Model':<25} {'Mean Accuracy':<15} {'Std Deviation'}\n")
    f.write("-" * 55 + "\n")
    for _, row in results_df.iterrows():
        f.write(f"{row['Model']:<25} {row['Mean Accuracy']*100:>6.2f}%         ±{row['Std Dev']*100:>5.2f}%\n")

print(f"\nResults saved to {out_path}")


# Generate Visualizations

Run this cell to regenerate the charts used in the thesis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

BASE_DIR = r"f:\CodeBlocks20.03-AnxietyMonitor"
DATA_FILE = os.path.join(BASE_DIR, "data", "processed", "master_dataset.csv")
VIS_DIR = os.path.join(BASE_DIR, "results", "visualizations")
os.makedirs(VIS_DIR, exist_ok=True)

# Style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
})

print("Loading data...")
df = pd.read_csv(DATA_FILE)
df['survey_reported_stress'] = pd.to_numeric(df['survey_reported_stress'], errors='coerce')

features = ['typing_speed_wpm', 'latency_variance_ms', 'pause_ratio',
            'backspace_rate', 'idle_ratio', 'focus_switches',
            'compile_success_rate', 'session_fragmentation', 'anxiety_score']
for col in features + ['survey_reported_stress']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=features + ['survey_reported_stress'])

student_agg = df.groupby(['student_id', 'session_name']).agg({
    'typing_speed_wpm': 'mean', 'latency_variance_ms': 'mean',
    'pause_ratio': 'mean', 'backspace_rate': 'mean', 'idle_ratio': 'mean',
    'focus_switches': 'max', 'compile_success_rate': 'mean',
    'session_fragmentation': 'mean', 'anxiety_score': 'mean',
    'survey_reported_stress': 'first'
}).reset_index()

print(f"Total students: {len(student_agg)}")

# ─────────────────────────────────────────────
# CHART 1: Scatter Plot - System Score vs Survey
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
palette = {'1st_ct': '#4C72B0', '2nd_ct': '#DD8452', '3rd_ct': '#55A868'}
for session, grp in student_agg.groupby('session_name'):
    ax.scatter(grp['anxiety_score'], grp['survey_reported_stress'],
               label=session.replace('_', ' ').title(),
               color=palette.get(session, 'gray'), s=80, alpha=0.8, edgecolors='white', linewidth=0.5)

# Regression line
m, b = np.polyfit(student_agg['anxiety_score'], student_agg['survey_reported_stress'], 1)
x_line = np.linspace(student_agg['anxiety_score'].min(), student_agg['anxiety_score'].max(), 100)
ax.plot(x_line, m*x_line + b, color='#c0392b', linewidth=2, linestyle='--', label='Trend Line')

pearson_r, p_p = pearsonr(student_agg['anxiety_score'], student_agg['survey_reported_stress'])
spearman_r, p_s = spearmanr(student_agg['anxiety_score'], student_agg['survey_reported_stress'])

ax.set_title(f'System Anxiety Score vs. Self-Reported Stress\n(N=44 students | Pearson r={pearson_r:.3f}, p={p_p:.4f} | Spearman ρ={spearman_r:.3f}, p={p_s:.4f})', fontsize=12)
ax.set_xlabel('Average Computed Anxiety Score (0–100)')
ax.set_ylabel('Self-Reported Stress Level (1–5)')
ax.set_yticks([1, 2, 3, 4, 5])
ax.legend(title='Session')
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'anxiety_vs_stress_scatter.png'), bbox_inches='tight')
plt.close()
print("[OK] Saved: anxiety_vs_stress_scatter.png")

# ─────────────────────────────────────────────
# CHART 2: Correlation Heatmap
# ─────────────────────────────────────────────
heatmap_cols = ['typing_speed_wpm', 'latency_variance_ms', 'pause_ratio',
                'backspace_rate', 'idle_ratio', 'focus_switches',
                'compile_success_rate', 'session_fragmentation',
                'anxiety_score', 'survey_reported_stress']
labels = ['Typing Speed\n(WPM)', 'Latency\nVariance (ms)', 'Pause\nRatio',
          'Backspace\nRate (%)', 'Idle\nRatio', 'Focus\nSwitches',
          'Compile\nSuccess Rate', 'Session\nFragmentation',
          'System\nAnxiety Score', 'Survey\nStress (1-5)']

corr = df[heatmap_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            xticklabels=labels, yticklabels=labels, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Behavioral Metrics Correlation Heatmap\n(Based on 8,505 rows across 44 students — 3 Class Tests)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'correlation_heatmap.png'), bbox_inches='tight')
plt.close()
print("[OK] Saved: correlation_heatmap.png")

# ─────────────────────────────────────────────
# CHART 3: ML Model Comparison Bar (No SMOTE)
# ─────────────────────────────────────────────
feat_cols = ['typing_speed_wpm', 'latency_variance_ms', 'pause_ratio',
             'backspace_rate', 'idle_ratio', 'focus_switches',
             'compile_success_rate', 'session_fragmentation']
X = student_agg[feat_cols]
y_class = (student_agg['survey_reported_stress'] >= 4).astype(int)
baseline = y_class.mean()

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM (Linear)': SVC(kernel='linear', random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=3),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=42)
results_plain = []
print("\nRunning 10x10 Repeated CV (No SMOTE)...")
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    cv = cross_validate(pipe, X, y_class, cv=rskf, scoring=['accuracy'], n_jobs=-1)
    acc = np.mean(cv['test_accuracy'])
    std = np.std(cv['test_accuracy'])
    results_plain.append({'Model': name, 'Accuracy': acc, 'Std': std})
    print(f"  {name}: {acc:.3f} ± {std:.3f}")

df_plain = pd.DataFrame(results_plain).sort_values('Accuracy', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if a > baseline else '#e74c3c' for a in df_plain['Accuracy']]
bars = ax.barh(df_plain['Model'], df_plain['Accuracy'], xerr=df_plain['Std'], color=colors, edgecolor='white', height=0.6, error_kw={'ecolor': '#34495e', 'capsize': 4, 'elinewidth': 1.5})
ax.axvline(x=baseline, color='#e74c3c', linestyle='--', linewidth=2, label=f'Imbalanced Baseline ({baseline:.1%})')
for bar, val in zip(bars, df_plain['Accuracy']):
    ax.text(val + 0.015, bar.get_y() + bar.get_height()/2, f'{val:.1%}', va='center', fontsize=10, fontweight='bold', color='#2c3e50')
ax.set_xlabel('Mean Accuracy over 100 Runs')
ax.set_title('ML Model Comparison — 10x10 Repeated CV (Without SMOTE)\n44 Students | Predict High Stress (≥4) | Error Bars = Std Deviation', fontsize=12)
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'model_comparison_bar.png'), bbox_inches='tight')
plt.close()
print("[OK] Saved: model_comparison_bar.png")

# ─────────────────────────────────────────────
# CHART 4: ML Model Comparison Bar (SMOTE)
# ─────────────────────────────────────────────
smote = SMOTE(k_neighbors=3, random_state=42)
results_smote = []
print("\nRunning 10x10 Repeated CV (With SMOTE)...")
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('smote', smote), ('clf', model)])
    cv = cross_validate(pipe, X, y_class, cv=rskf, scoring=['accuracy'], n_jobs=-1)
    acc = np.mean(cv['test_accuracy'])
    std = np.std(cv['test_accuracy'])
    results_smote.append({'Model': name, 'Accuracy': acc, 'Std': std})
    print(f"  {name}: {acc:.3f} ± {std:.3f}")

df_smote = pd.DataFrame(results_smote).sort_values('Accuracy', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#3498db' if a > 0.5 else '#e74c3c' for a in df_smote['Accuracy']]
bars = ax.barh(df_smote['Model'], df_smote['Accuracy'], xerr=df_smote['Std'], color=colors, edgecolor='white', height=0.6, error_kw={'ecolor': '#34495e', 'capsize': 4, 'elinewidth': 1.5})
ax.axvline(x=0.5, color='#e74c3c', linestyle='--', linewidth=2, label='Random Guessing Baseline (50%)')
for bar, val in zip(bars, df_smote['Accuracy']):
    ax.text(val + 0.015, bar.get_y() + bar.get_height()/2, f'{val:.1%}', va='center', fontsize=10, fontweight='bold', color='#2c3e50')
ax.set_xlabel('Mean Accuracy over 100 Runs')
ax.set_title('ML Model Comparison — 10x10 Repeated CV (With SMOTE Balancing)\n44 Students | Balanced Classes | Error Bars = Std Deviation', fontsize=12)
ax.set_xlim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'smote_model_comparison_bar.png'), bbox_inches='tight')
plt.close()
print("[OK] Saved: smote_model_comparison_bar.png")

# ─────────────────────────────────────────────
# CHART 5: Stress Distribution by Session
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Distribution of survey stress levels
stress_counts = student_agg['survey_reported_stress'].value_counts().sort_index()
colors_bar = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#c0392b']
axes[0].bar(stress_counts.index, stress_counts.values, color=colors_bar[:len(stress_counts)], edgecolor='white')
axes[0].set_title('Distribution of Self-Reported Stress Levels\n(N=44 students)')
axes[0].set_xlabel('Stress Level (1=Low, 5=High)')
axes[0].set_ylabel('Number of Students')
axes[0].set_xticks([1,2,3,4,5])
for i, v in zip(stress_counts.index, stress_counts.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Right: Avg anxiety score by session
session_avg = student_agg.groupby('session_name')['anxiety_score'].mean().reset_index()
session_avg['session_name'] = session_avg['session_name'].str.replace('_', ' ').str.title()
session_colors = ['#4C72B0', '#DD8452', '#55A868']
axes[1].bar(session_avg['session_name'], session_avg['anxiety_score'], color=session_colors, edgecolor='white')
axes[1].set_title('Average System Anxiety Score by Session\n(N=44 students across 3 Class Tests)')
axes[1].set_xlabel('Class Test Session')
axes[1].set_ylabel('Average Anxiety Score (0–100)')
axes[1].set_ylim(0, 100)
for i, v in enumerate(session_avg['anxiety_score']):
    axes[1].text(i, v + 1, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'stress_distribution_by_session.png'), bbox_inches='tight')
plt.close()
print("[OK] Saved: stress_distribution_by_session.png")

# ─────────────────────────────────────────────
# CHART 6: Feature Importance (Random Forest)
# ─────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.preprocessing import StandardScaler as SS

scaler = SS()
X_scaled = scaler.fit_transform(X)
rf = RFC(n_estimators=200, random_state=42)
rf.fit(X_scaled, y_class)
importances = pd.DataFrame({'Feature': feat_cols, 'Importance': rf.feature_importances_})
importances = importances.sort_values('Importance', ascending=True)
feat_labels = {
    'typing_speed_wpm': 'Typing Speed (WPM)',
    'latency_variance_ms': 'Latency Variance (ms)',
    'pause_ratio': 'Pause Ratio',
    'backspace_rate': 'Backspace Rate (%)',
    'idle_ratio': 'Idle Ratio',
    'focus_switches': 'Focus Switches',
    'compile_success_rate': 'Compile Success Rate',
    'session_fragmentation': 'Session Fragmentation'
}
importances['Feature'] = importances['Feature'].map(feat_labels)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importances['Feature'], importances['Importance'], color='#3498db', edgecolor='white')
for bar, val in zip(bars, importances['Importance']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=10)
ax.set_xlabel('Feature Importance Score')
ax.set_title('Random Forest — Feature Importances for Anxiety Prediction\n(Trained on 44 students | All 3 Class Tests)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'feature_importance.png'), bbox_inches='tight')
plt.close()
print("[OK] Saved: feature_importance.png")

print("[DONE] All visualizations regenerated successfully!")
print(f"Saved to: {VIS_DIR}")
